In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=20, edgecolor='black')
plt.title('distribution of Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
categorical_column = df.select_dtypes(include='object').columns
for col in categorical_column:
    df[col] = df[col].fillna('Unknown')

In [ ]:
check_missing_values(df)

In [ ]:
df["Courier_Experience_yrs"] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0])

In [ ]:
df["Delivery_Time"] = df['Delivery_Time'].fillna(df['Delivery_Time'].mode()[0])

In [ ]:
check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# check again
check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
  print(f"Encoding column: {col}")
  ohe = OneHotEncoder(sparse_output=False)

  df = df.join(
    pd.DataFrame(
        ohe.fit_transform(df[[col]]),
        columns=ohe.get_feature_names_out([col]),
        index=df.index
    )
    ).drop(columns=[col])

df.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
df["Delivery_Time"].value_counts()

In [ ]:
## The target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
feature_cols = ["Distance_km", "Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type", "Preparation_Time_min",
                "Courier_Experience_yrs", "Delivery_Time"
                ]

In [ ]:
# Task 2,3,4,5: Write your code here:

# we will use StratifiedKFold because the imbalance
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor


In [ ]:
from sklearn.metrics import mean_absolute_error as sklearn_mae

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)


    model = RandomForestRegressor(n_estimators=200)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae_scores.append(sklearn_mae(y_test, y_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred ")
plt.title("Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: